# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [1]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

In [ ]:
sql("SELECT * FROM shop.catalog LIMIT 5")

,brand,category,created_at,description,is_active,price_eur,product_id,subcategory
0,Vedo,Sports & Outdoors,2026-01-21 07:00:00.000000000,Vedo Yoga Mats with organic ingredients and ex...,True,14.99,p_001897,Yoga Mats
1,Zore,Fashion,2025-11-06 07:00:00.000000000,"From Zore (Jackets): wireless connectivity, a ...",True,4.99,p_00196b,Jackets
2,Sanine,Electronics,2025-01-28 07:00:00.000000000,Sanine Laptop Sleeves with a luxury feel and a...,True,57.99,p_001a20,Laptop Sleeves
3,Vebexa,Sports & Outdoors,2024-09-18 07:00:00.000000000,Vebexa Running Shoes with built for profession...,True,29.99,p_001d7f,Running Shoes
4,Nibe,Home & Kitchen,2025-10-15 07:00:00.000000000,Nibe Cutting Boards with a minimalist design a...,True,26.99,p_0032c5,Cutting Boards
5,Nika,Beauty & Health,2025-11-04 07:00:00.000000000,Nika Vitamins with a waterproof shell and a ha...,True,46.99,p_003840,Vitamins
6,Zofiri,Fashion,2026-07-08 07:00:00.000000000,"Zofiri Socks line, known for great value for m...",True,11.99,p_004caa,Socks
7,Rixa,Pets,2025-09-18 07:00:00.000000000,Rixa Dog Toys with a minimalist design and a s...,True,226.99,p_00675e,Dog Toys
8,Tixa,Toys & Baby,2025-08-30 07:00:00.000000000,"Strollers by Tixa, featuring wireless connecti...",True,34.99,p_007935,Strollers
9,Luzo,Home & Kitchen,2025-03-04 07:00:00.000000000,From Luzo (Coffee Machines): compact and light...,True,1.99,p_008500,Coffee Machines


In [4]:
sql("SELECT * FROM shop.requests ORDER BY ts DESC LIMIT 5")

,context.device,context.page,context.region,is_new_user,recent_views,request_id,ts,user_id
0,android,product,Madrid,True,"[{'product_id': 'p_7c9073', 'ts': '2026-11-14 ...",r_194107ce7436ecb4,2026-11-14 12:16:18.011000000,u_56c60b0f
1,android,category,Andalucía,False,"[{'product_id': 'p_e4858b', 'ts': '2026-11-14 ...",r_8c86e470af105869,2026-11-14 12:16:17.723000000,u_6dc37df2
2,web,category,País Vasco,True,"[{'product_id': 'p_56b3b0', 'ts': '2026-11-14 ...",r_2e407c3b4439099f,2026-11-14 12:16:17.340000000,u_9ffa34b7
3,ios,home,Catalunya,False,[],r_2d7bf0307091640e,2026-11-14 12:16:17.024000000,u_10070592
4,web,category,Andalucía,False,"[{'product_id': 'p_85673e', 'ts': '2026-11-14 ...",r_212a7f4a39dd501b,2026-11-14 12:16:16.678000000,u_760217a3


In [5]:
sql("SELECT * FROM shop.history_sessions LIMIT 3")

,clicked,context.device,context.page,context.region,propensity,purchased,recent_views,request_id,revenue_eur,shown,ts,user_id
0,None,web,product,Catalunya,"[0.218886, 0.002833, 0.11516, 0.321647, 0.0292...",None,"[{'product_id': 'p_159190', 'ts': '2026-09-15 ...",r_50309edc92778291,0,"[p_8ca8ed, p_8ac75a, p_e2c959, p_2c88d5, p_8fb...",2026-09-15 07:00:01.456000000,u_20cbc5d5
1,None,web,product,Madrid,"[0.032807, 0.011809, 0.008224, 0.025852, 0.290...",None,"[{'product_id': 'p_f3c09e', 'ts': '2026-09-15 ...",r_7a441a0fb217005e,0,"[p_032d24, p_40bbb4, p_66f831, p_d245de, p_adc...",2026-09-15 07:00:13.975000000,u_cd609f53
2,None,android,category,Aragón,"[0.009835, 0.271305, 0.005089, 0.050632, 0.165...",None,"[{'product_id': 'p_f41d0b', 'ts': '2026-09-15 ...",r_7fa58169a04e436c,0,"[p_9c2b8b, p_adc3e6, p_883ab9, p_07b81b, p_09a...",2026-09-15 07:00:19.633000000,u_5a69bbf5


In [6]:
df = sql("""
SELECT e.region::String AS region, c.category::String AS category, count() AS purchases
FROM shop.history_events AS e
INNER JOIN shop.catalog AS c ON c.product_id::String = e.product_id::String
WHERE e.event_type = 'purchase'
GROUP BY region, category
""")
df.pivot_table(index="region", columns="category", values="purchases")

category,Beauty & Health,Books & Media,Computers & Gaming,Electronics,Fashion,Home & Garden,Home & Kitchen,Pets,Sports & Outdoors,Toys & Baby
region,,,,,,,,,,
Andalucía,8163.0,29472.0,12033.0,14084.0,13511.0,4402.0,7298.0,6071.0,11707.0,17487.0
Aragón,1165.0,4262.0,1547.0,1715.0,2007.0,613.0,842.0,1024.0,1891.0,2385.0
Canarias,1866.0,5757.0,2783.0,3624.0,3137.0,1021.0,1673.0,1311.0,2969.0,4351.0
Castilla y León,2331.0,7778.0,2718.0,3440.0,3004.0,1137.0,1688.0,1890.0,3203.0,4379.0
Catalunya,6677.0,24339.0,9378.0,11220.0,9714.0,3416.0,5125.0,5819.0,10901.0,14513.0
Comunitat Valenciana,4673.0,16836.0,6262.0,6891.0,6148.0,2170.0,3288.0,3625.0,7483.0,10051.0
Galicia,2783.0,8853.0,3322.0,3859.0,3811.0,1373.0,1824.0,2132.0,4006.0,4791.0
Madrid,5846.0,22187.0,7761.0,9555.0,8609.0,2986.0,4403.0,4835.0,10581.0,12162.0
Murcia,1334.0,5752.0,1995.0,2399.0,2154.0,753.0,1185.0,987.0,1581.0,2870.0


In [7]:
sql("""
SELECT fallback_reason::String AS reason, count() AS n,
       quantile(0.95)(latency_ms::Float64) AS p95_ms
FROM team_15.feedback
WHERE ts > now() - INTERVAL 15 MINUTE AND request_id != '__init__'
GROUP BY reason
""")

,reason,n,p95_ms
0,invalid,1,2332.00
1,,2700,1759.15


In [ ]:
t = df.pivot_table(index="region", columns="category", values="purchases")
shares = t.div(t.sum(axis=1), axis=0) * 100
shares.round(1)

category,Beauty & Health,Books & Media,Computers & Gaming,Electronics,Fashion,Home & Garden,Home & Kitchen,Pets,Sports & Outdoors,Toys & Baby
region,,,,,,,,,,
Andalucía,6.6,23.7,9.7,11.3,10.9,3.5,5.9,4.9,9.4,14.1
Aragón,6.7,24.4,8.9,9.8,11.5,3.5,4.8,5.9,10.8,13.7
Canarias,6.5,20.2,9.8,12.7,11.0,3.6,5.9,4.6,10.4,15.3
Castilla y León,7.4,24.6,8.6,10.9,9.5,3.6,5.3,6.0,10.1,13.9
Catalunya,6.6,24.1,9.3,11.1,9.6,3.4,5.1,5.8,10.8,14.4
Comunitat Valenciana,6.9,25.0,9.3,10.2,9.1,3.2,4.9,5.4,11.1,14.9
Galicia,7.6,24.1,9.0,10.5,10.4,3.7,5.0,5.8,10.9,13.0
Madrid,6.6,25.0,8.7,10.7,9.7,3.4,5.0,5.4,11.9,13.7
Murcia,6.3,27.4,9.5,11.4,10.3,3.6,5.6,4.7,7.5,13.7


## Data

In [ ]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

In [ ]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

In [ ]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [ ]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)

def coview(row):
    """Products co-viewed with this session's views (most recent first), then the baseline."""
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

POLICIES = {"baseline": baseline, "coview": coview}

## Evaluation

In [ ]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.